In [1]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher, tqdm
StreamlitPatcher().jupyter()  # register streamlit with jupyter-compatible wrappers
import sys; sys.path.append('..')
from osp import *
pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 20

In [2]:
# pprint(get_sent_html(get_sent_obj("The old world is dying, and the new world struggles to be born."),show_labels=True))

In [3]:
doc = get_nlp_doc(newtxt)
sent = doc.sentences[0]
# sent.text

In [4]:
HTML(get_sent_html(sent,show_labels=True,highlight_word_id=15))

In [5]:
get_feat_group_egs('deprel_mark')

({}, {})

In [6]:
def get_slices_feats(slice_ids):
    out = []
    for slice_id in slice_ids:
        res_d = STASH_SLICE_FEATS.get(slice_id, None)
        if res_d:
            out.append(res_d)
    return pd.DataFrame(out, index=slice_ids).rename_axis('slice_id')

In [7]:
get_slices_feats(get_slice_ids('discipline in ["Philosophy"]'))

,pos_IN,pos_DT,pos_NN,pos_PRP,pos_MD,pos_VB,pos_TO,pos_VBG,pos_VBZ,pos_JJ,...,phrase_:,phrase_RRC,phrase_``,pos_ADD,pos_GW,phrase_NAC,phrase_ADD,deprel_goeswith,phrase_$,deprel_obl:tmod
slice_id,,,,,,,,,,,,,,,,,,,,,
phil/10.2307/40231690__03,135.299219,108.412836,167.389419,57.241977,21.682567,49.436253,20.815265,13.009540,37.294016,70.251518,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/40231533__01,144.859813,91.900312,98.909657,40.498442,15.576324,31.931464,9.345794,13.239875,56.853583,70.872274,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/40230622__01,138.599106,115.499255,159.463487,26.825633,14.903130,30.551416,8.941878,6.706408,63.338301,58.867362,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/40231425__01,142.381349,98.251457,138.218152,34.970858,14.154871,43.297252,19.150708,23.313905,51.623647,62.447960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/40231425__04,131.059246,119.389587,178.635548,26.032316,21.543986,51.166966,15.260323,17.055655,61.041293,67.324955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
phil/10.2307/20123425__04,122.267206,64.777328,93.117409,36.437247,21.862348,62.348178,20.242915,19.433198,38.866397,84.210526,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/20124042__02,106.215578,139.260425,146.341463,36.978757,12.588513,34.618411,14.162077,5.507474,62.942565,56.648308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/20129720__04,152.653061,113.469388,167.346939,33.469388,12.244898,35.918367,15.510204,13.061224,45.714286,62.857143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
get_feat_group_egs(['deprel_mark'], groups=COMPARISONS[2])

({}, {})

In [9]:
def get_slices_feats(slice_ids):
    out = []
    for slice_id in slice_ids:
        res_d = STASH_SLICE_FEATS.get(slice_id, None)
        if res_d:
            out.append(res_d)
    return pd.DataFrame(out, index=slice_ids).rename_axis('slice_id')

def get_feat_egs(df_feats, feats, num_egs=10):
    out_feat2egs = []
    for feat in feats:
        feat_egs = []
        done_words = set()
        if feat not in df_feats.columns:
            continue
        top_slices = df_feats.sort_values(by=feat, ascending=False)
        for slice_id in top_slices.index:
            egs_feat2word2eg = STASH_FEAT2WORD2EG.get(slice_id, {})
            if not egs_feat2word2eg:
                continue
            word2eg = egs_feat2word2eg.get(feat, egs_feat2word2eg.get(feat.split('_',1)[-1], {}))
            word2eg = {k:v for k,v in word2eg.items() if k not in done_words}
            if not word2eg:
                continue
            
            word = random.choice(list(word2eg.keys()))
            eg = word2eg[word]
            feat_egs.append({'feat': feat, 'word': word, 'eg': eg, 'slice_id': slice_id})
            done_words.add(word)
            if len(done_words)>=num_egs:
                out_feat2egs.extend(feat_egs)
                break
    return pd.DataFrame(out_feat2egs)

def get_feat_group_egs(feats, groups=None, num_egs=10):
    # from .slices import get_slice_ids
    if isinstance(feats, str):
        feats = [feats]
    if groups is None:
        groups = COMPARISONS[0]
    
    name1,query1 = groups[0]
    name2,query2 = groups[1]
    
    slice_ids1 = get_slice_ids(query1)
    slice_ids2 = get_slice_ids(query2)
    
    df_feats1 = get_slices_feats(slice_ids1)
    df_feats2 = get_slices_feats(slice_ids2)

    odf1 = get_feat_egs(df_feats1, feats, num_egs=num_egs)
    odf2 = get_feat_egs(df_feats2, feats, num_egs=num_egs)

    return pd.concat([odf1.assign(group=name1), odf2.assign(group=name2)])

In [10]:
dfx=get_current_feat_weights().sort_values('weight',ascending=False)
dfx

,weight,mean_Literature,mean_Philosophy,run,weight_z
feature,,,,,
deprel_mark,0.852745,-0.719181,0.199986,4.5,3.473451
deprel_cop,0.645030,-0.654353,0.328836,4.5,2.629497
phrase_ROOT,0.643411,-0.385347,0.226749,4.5,2.622918
deprel_ccomp,0.582592,-0.565852,0.103643,4.5,2.375812
deprel_advmod,0.477965,-0.430795,0.158886,4.5,1.950707
...,...,...,...,...,...
ttr_OTHER,-0.452262,0.833292,-0.374745,4.5,-1.828829
pos_FW,-0.559887,0.402330,-0.137235,4.5,-2.266110
ttr_NOUN,-0.561027,0.744833,-0.101287,4.5,-2.270744


In [11]:
feats = dfx.index.tolist()

In [12]:
a = get_feat_group_egs(feats)
a

KeyboardInterrupt: 

In [ ]:
get_current_feat_weights().sort_values('weight_z',ascending=False)

In [ ]:
def extract_feat_examples(doc):
    sents = doc.sentences
    random.shuffle(sents)
    egs = defaultdict(list)
    for sent in sents:
        for word in sent.words:
            for feat_type in ['deprel','pos']:
                feat_val = word.deprel if feat_type == 'deprel' else word.xpos
                if feat_val:
                    feat = f'{feat_type}_{feat_val}'
                    if feat not in egs:
                        egs[feat] = 


def gen_feat_examples():
    df_all_z = get_all_feats(normalize=False)
    df_all_raw = get_all_feats(normalize=False)
    for id,row in df_all_raw.iterrows():
        row_feats = row[row>0].index
        row_feats = [x for x in row_feats if x.split('_',1)[0] in ['deprel','pos']]
        if not row_feats: continue

        docstr = STASH_SLICES_NLP.get(id,None)
        if not docstr: continue

        doc = stanza.Document.from_serialized(docstr)
        feat_egs = extract_feat_examples(doc)
        return feat_egs
        
        


In [ ]:
for id,docstr in STASH_SLICES_NLP.items():
    doc = stanza.Document.from_serialized(docstr)
    break

In [ ]:
sents = doc.sentences
random.shuffle(sents)
sents[0].to_dict()

In [ ]:
gen_feat_examples()

In [ ]:
extracted_examples = extract_examples()
extracted_examples


In [ ]:
slice_ids = get_slice_ids(['phil/10.2307/40231690'])

In [ ]:
get_slices_feats(['phil/10.2307/40231690'])